In [1]:
pip install sentence-transformers faiss-cpu numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.5 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================
DB_DIR = "dense_index_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss_index.bin")
METADATA_PATH = os.path.join(DB_DIR, "paragraph_metadata.json")

os.makedirs(DB_DIR, exist_ok=True)

# Define standard embedding size for sentence-transformers/all-MiniLM-L6-v2
EMBEDDING_DIM = 384

print("🤖 Loading sentence-transformers/all-MiniLM-L6-v2 model...")
# Automatically downloads and caches the model on first fresh run
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ==========================================
# 2. CREATE DEV-SET PARAGRAPH POOL
# ==========================================
# Mock Dev Corpus representing a document dataset needing compilation into a pool
raw_documents = {
    "doc_1": """Dense vector retrieval leverages neural network encoders to map texts into a continuous vector space. This allows downstream applications to perform semantic matches instead of exact string token matches. FAISS is perfectly suited for managing these nearest neighbor workloads efficiently.""",
    "doc_2": """FAISS (Facebook AI Similarity Search) is an optimized C++ library with elegant Python wrappers designed specifically for ultra-fast dense vector clustering and geometric similarity calculations. It balances memory allocation constraints against extreme processing speeds.""",
    "doc_3": """When preprocessing paragraph pools for production development sets, always ensure clean line breaks are standard. Strip stray markdown artifacts or excessive white space from raw text strings to prevent corrupt tokenizing behavior inside small transformers like MiniLM."""
}

paragraph_pool = []
paragraph_metadata = []

print("📝 Processing raw text into paragraph pools...")
for doc_id, text in raw_documents.items():
    # Split text by paragraph barriers (using double linebreaks or custom chunks as logic commands)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    for idx, para in enumerate(paragraphs):
        paragraph_pool.append(para)
        paragraph_metadata.append({
            "global_id": len(paragraph_pool) - 1,
            "source_doc": doc_id,
            "local_chunk_index": idx,
            "text": para
        })

print(f"✅ Generated a dev pool containing {len(paragraph_pool)} unique paragraphs.")

# ==========================================
# 3. COMPUTE EMBEDDINGS & BUILD FAISS INDEX
# ==========================================
print("⚡ Generating dense embeddings with MiniLM-L6-v2...")
# Extract embeddings (Returns raw NumPy arrays)
embeddings = model.encode(paragraph_pool, show_progress_bar=True, convert_to_numpy=True)

# Normalize vectors for Cosine Similarity (Inner Product Indexing)
faiss.normalize_L2(embeddings)

print("🏗️ Initializing FAISS Index Flat IP (Inner Product)...")
# IndexFlatIP handles cosine similarity calculations directly on normalized vectors
index = faiss.IndexFlatIP(EMBEDDING_DIM)

# Register the text embeddings to the geometric matrix
index.add(embeddings)
print(f"📊 Vector footprint currently mapped in FAISS: {index.ntotal}")

# ==========================================
# 4. SAVE AND UTILITY EXPORT
# ==========================================
print("💾 Persisting FAISS Index and mapping configuration metadata to disk...")
faiss.write_index(index, INDEX_PATH)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(paragraph_metadata, f, ensure_ascii=False, indent=4)

print(f"✨ Build Complete! Files stored safely inside current workspace at: '{DB_DIR}/'\n")

# ==========================================
# 5. DRY-RUN RE-LOAD & QUERY VALIDATION
# ==========================================
print("🧪 Executing isolated query test against newly built index...")
# Read files straight from directory to ensure fresh clone works immediately
loaded_index = faiss.read_index(INDEX_PATH)
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    loaded_meta = json.load(f)

test_query = "How fast can FAISS perform geometric searches?"
query_vector = model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(query_vector)

# Query top K matches
K = 1
scores, indices = loaded_index.search(query_vector, K)

print("\n--- Search Query Results Summary ---")
print(f"Query: \"{test_query}\"")
for rank in range(K):
    meta_idx = indices[0][rank]
    print(f"Top Hit Score: {scores[0][rank]:.4f}")
    print(f"Source Document Context: {loaded_meta[meta_idx]['source_doc']}")
    print(f"Extracted String: \"{loaded_meta[meta_idx]['text']}\"\n")


🤖 Loading sentence-transformers/all-MiniLM-L6-v2 model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📝 Processing raw text into paragraph pools...
✅ Generated a dev pool containing 3 unique paragraphs.
⚡ Generating dense embeddings with MiniLM-L6-v2...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🏗️ Initializing FAISS Index Flat IP (Inner Product)...
📊 Vector footprint currently mapped in FAISS: 3
💾 Persisting FAISS Index and mapping configuration metadata to disk...
✨ Build Complete! Files stored safely inside current workspace at: 'dense_index_db/'

🧪 Executing isolated query test against newly built index...

--- Search Query Results Summary ---
Query: "How fast can FAISS perform geometric searches?"
Top Hit Score: 0.5252
Source Document Context: doc_2
Extracted String: "FAISS (Facebook AI Similarity Search) is an optimized C++ library with elegant Python wrappers designed specifically for ultra-fast dense vector clustering and geometric similarity calculations. It balances memory allocation constraints against extreme processing speeds."



In [3]:
%%writefile build_pipeline.py

Writing build_pipeline.py


In [8]:
!python build_pipeline.py


=== 📊 Hard Negative Similarity Distribution Report ===
Total Queries Analyzed (N): 1000
Negative Samples Per Query (k): 16
-------------------------------------------------------
Global Mean Negative Score:         0.3742
Mean of Max-Negatives (Hardest):    0.5296
95th Percentile of Max-Negatives:   0.5489
-------------------------------------------------------
Mean Score Margin (Gold vs Max-Neg): 0.2184
⚠️ Label Leakage Risk Ratio (Margin < 0.05): 0.00%
[VERIFIED] Score distributions fall into a healthy contrastive learning sweet spot.

=== ⚙️ PyTorch Tokenized Batch Footprint ===
Anchor Input Shape:   torch.Size([2, 5])
Positive Input Shape: torch.Size([2, 7])
Negative Input Shape: torch.Size([32, 13])


#Test code inside the build_pipeline.py
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. SETUP
DB_DIR = "dense_index_db"
INDEX_PATH = os.path.join(DB_DIR, "faiss_index.bin")
METADATA_PATH = os.path.join(DB_DIR, "paragraph_metadata.json")
os.makedirs(DB_DIR, exist_ok=True)

print("🤖 Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. MOCK DATA
raw_documents = {
    "doc_1": "Dense vector retrieval leverages neural network encoders to map texts into a continuous vector space.",
    "doc_2": "FAISS (Facebook AI Similarity Search) is an optimized library designed for fast similarity calculations.",
}

paragraph_pool = []
paragraph_metadata = []

for doc_id, text in raw_documents.items():
    paragraph_pool.append(text)
    paragraph_metadata.append({"global_id": len(paragraph_pool) - 1, "source_doc": doc_id, "text": text})

# 3. EMBED & INDEX
print("⚡ Embedding...")
embeddings = model.encode(paragraph_pool, convert_to_numpy=True)
faiss.normalize_L2(embeddings)

index = faiss.IndexFlatIP(384)
index.add(embeddings)

# 4. SAVE
faiss.write_index(index, INDEX_PATH)
with open(METADATA_PATH, "w") as f:
    json.dump(paragraph_metadata, f, indent=4)

# 5. TEST QUERY
test_query = "Is Facebook library quick at vector calculations?"
query_vector = model.encode([test_query], convert_to_numpy=True)
faiss.normalize_L2(query_vector)

scores, indices = index.search(query_vector, 1)
print(f"\n🔍 Query: '{test_query}'")
print(f"🎯 Match: '{paragraph_metadata[indices[0][0]]['text']}' (Score: {scores[0][0]:.4f})")

# **Review prompt template and answer-extraction specification**

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

class TripletHardNegativeDataset(Dataset):
    def __init__(self, data_list):
        """
        Expected schema for data_list:
        [
            {
                "anchor": "How to optimize retrieval speeds in FAISS?",
                "positive": "FAISS leverages highly optimized low-level C++ wrappers to allocate memory...",
                "negatives": ["Passage 1", "Passage 2", ..., "Passage 16"] # Array must contain exactly 16 strings
            },
            ...
        ]
        """
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Return item container containing 1 Anchor text, 1 Gold target, and 16 Hard Negatives
        return {
            "anchor": item["anchor"],
            "positive": item["positive"],
            "negatives": item["negatives"] # List of strings (Len: 16)
        }

class BiEncoderCollateFn:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", max_length=256):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = max_length

    def __call__(self, batch):
        anchors = [item["anchor"] for item in batch]
        positives = [item["positive"] for item in batch]

        # Flatten the 2D matrix of negative strings into a 1D sequence for batch tokenization
        flat_negatives = []
        for item in batch:
            flat_negatives.extend(item["negatives"])

        # Unified lambda wrapper for tokenization configurations
        tokenize_capsule = lambda texts: self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "anchor_input": tokenize_capsule(anchors),
            "positive_input": tokenize_capsule(positives),
            # Shape output: [BatchSize * 16, SeqLen].
            # Reconstruct tensor shape inside training step using: .view(BatchSize, 16, -1)
            "negative_input": tokenize_capsule(flat_negatives),
            "batch_size": len(batch),
            "negatives_per_query": len(batch[0]["negatives"])
        }

# ==========================================
# 🧪 Pipe Integration Test (Dry-Run Verification)
# ==========================================
if __name__ == "__main__":
    # Construct 2 mock samples for validation testing
    mock_dataset_source = [
        {
            "anchor": "Query Text Alpha",
            "positive": "Gold Positive Document Content Alpha",
            "negatives": [f"Mined Hard Adversarial Negative Doc {i} for Alpha" for i in range(16)]
        },
        {
            "anchor": "Query Text Beta",
            "positive": "Gold Positive Document Content Beta",
            "negatives": [f"Mined Hard Adversarial Negative Doc {i} for Beta" for i in range(16)]
        }
    ]

    # Initialize the customized PyTorch pipeline components
    dataset = TripletHardNegativeDataset(mock_dataset_source)
    collator = BiEncoderCollateFn(model_name="sentence-transformers/all-MiniLM-L6-v2", max_length=128)

    dataloader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collator)

    # Validate resulting structural dimension configurations
    for batch_data in dataloader:
        print("\n=== ⚙️ PyTorch Tokenized Batch Footprint ===")
        print(f"Anchor Input Shape:   {batch_data['anchor_input']['input_ids'].shape}")   # Yields [2, SeqLen]
        print(f"Positive Input Shape: {batch_data['positive_input']['input_ids'].shape}") # Yields [2, SeqLen]
        print(f"Negative Input Shape: {batch_data['negative_input']['input_ids'].shape}") # Yields [32, SeqLen] (2 * 16 flattened)
        print("============================================")



=== ⚙️ PyTorch Tokenized Batch Footprint ===
Anchor Input Shape:   torch.Size([2, 5])
Positive Input Shape: torch.Size([2, 7])
Negative Input Shape: torch.Size([32, 13])


# **perform a margin analysis between the gold positive similarity scores and the mined Top-16 hard negatives**

In [5]:
import numpy as np
import pandas as pd

def generate_similarity_report(gold_scores, negative_scores_matrix):
    """
    Arguments:
    - gold_scores: Array of positive match scores per query (shape: [N])
    - negative_scores_matrix: Matrix of top-16 negative scores per query (shape: [N, 16])
    """
    df_neg = pd.DataFrame(negative_scores_matrix)

    print("=== 📊 Hard Negative Similarity Distribution Report ===")
    print(f"Total Queries Analyzed (N): {len(gold_scores)}")
    print(f"Negative Samples Per Query (k): {negative_scores_matrix.shape[1]}")
    print("-" * 55)

    # 1. Aggregate negative score distributions
    mean_neg = np.mean(negative_scores_matrix)
    max_neg_per_query = np.max(negative_scores_matrix, axis=1)

    print(f"Global Mean Negative Score:         {mean_neg:.4f}")
    print(f"Mean of Max-Negatives (Hardest):    {np.mean(max_neg_per_query):.4f}")
    print(f"95th Percentile of Max-Negatives:   {np.percentile(max_neg_per_query, 95):.4f}")

    # 2. Score Margin Analysis (Gold Score - Max Negative Score)
    margins = gold_scores - max_neg_per_query
    print("-" * 55)
    print(f"Mean Score Margin (Gold vs Max-Neg): {np.mean(margins):.4f}")

    # 3. Label Leakage Risk Audit (False Negative Risk Assessment)
    leakage_risk = np.sum(margins < 0.05) / len(margins) * 100
    print(f"⚠️ Label Leakage Risk Ratio (Margin < 0.05): {leakage_risk:.2f}%")
    if leakage_risk > 5.0:
        print("[WARNING] Critical negative scores are too close to gold targets. Consider raising the exclusion threshold.")
    else:
        print("[VERIFIED] Score distributions fall into a healthy contrastive learning sweet spot.")
    print("=" * 55)

# Simulate evaluation data matrix (N=1000 queries)
np.random.seed(42)
mock_gold = np.random.uniform(0.65, 0.85, 1000)
mock_negs = np.random.uniform(0.20, 0.55, (1000, 16)) # Mocked Top-16 scores
generate_similarity_report(mock_gold, mock_negs)


=== 📊 Hard Negative Similarity Distribution Report ===
Total Queries Analyzed (N): 1000
Negative Samples Per Query (k): 16
-------------------------------------------------------
Global Mean Negative Score:         0.3742
Mean of Max-Negatives (Hardest):    0.5296
95th Percentile of Max-Negatives:   0.5489
-------------------------------------------------------
Mean Score Margin (Gold vs Max-Neg): 0.2184
⚠️ Label Leakage Risk Ratio (Margin < 0.05): 0.00%
[VERIFIED] Score distributions fall into a healthy contrastive learning sweet spot.
